YOLOv11改进模型训练脚本
集成MMSA注意力机制

In [1]:
import os
from pathlib import Path
from ultralytics import YOLO
import yaml

In [2]:
# 路径配置
current_dir = Path.cwd()
project_root = current_dir / "ship_detection"
dataset_dir = project_root / "datasets" / "seaships"
results_dir = project_root / "results"
    
# 加载数据集配置
with open(dataset_dir / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)
    
print(f"数据集: {data_config['names']}")
print(f"类别数: {data_config['nc']}")

数据集: ['ship']
类别数: 1


In [3]:
def train_improved_model():
    custom_yaml = project_root / "custom_yolov11.yaml"
    model = YOLO(str(custom_yaml)).load("yolo11s.pt")
    
    # 开始训练
    results = model.train(
        data=str(dataset_dir / "data.yaml"),
        epochs=200,                # 适当增加轮数
        imgsz=512,
        batch=1,
        workers=1,
        device=0,                  
        project=str(results_dir / "improved"),
        name="mmsa_con",
        exist_ok=True,
        patience=30,
        save=True,
        save_period=10,
        plots=True,
        cache=False,
        amp=True,
        lr0=0.001,                  # 初始学习率
        lrf=0.01,                  # 最终学习率
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=7.5,                   # 边界框损失权重
        cls=0.5,                   # 分类损失权重
        dfl=1.5,                   # DFL损失权重
    )
    
    print("改进模型训练完成")
    return results

In [4]:
def evaluate_models():
    baseline_model_path = results_dir / "yolov11" / "baseline512" / "weights" / "best.pt"
    improved_model_path = results_dir / "improved" / "mmsa_con" / "weights" / "best.pt"
    path512 = results_dir / "improved" / "mmsa_512" / "weights" / "best.pt"

    if not baseline_model_path.exists():
        print("基准模型未找到，请先训练基准模型")
        return None, None
    
    baseline_model = YOLO(str(baseline_model_path))
    improved_model = YOLO(str(improved_model_path))
    model_512 = YOLO(str(path512))
    
    baseline_metrics = baseline_model.val(
        data=str(dataset_dir / "data.yaml"),
        split='test',
        batch=16,
        conf=0.25,
        iou=0.5
    )
    improved_metrics = improved_model.val(
        data=str(dataset_dir / "data.yaml"),
        split='test',
        batch=4,
        conf=0.25,
        iou=0.5
    )
    metrics512 = model_512.val(
        data=str(dataset_dir / "data.yaml"),
        split='test',
        batch=4,
        conf=0.25,
        iou=0.5
    )
    
    print("\n" + "="*50)
    print("模型性能对比")
    print("="*50)
    print(f"{'指标':<20} {'基准模型':<15} {'改进模型':<15} {'提升':<10}{'改进模型2':<15}")
    print("-"*50)
    print(f"{'mAP@0.5':<20} {baseline_metrics.box.map50:.4f}        {improved_metrics.box.map50:.4f}        {improved_metrics.box.map50 - baseline_metrics.box.map50:+.4f}      {metrics512.box.map50:.4f}")
    print(f"{'mAP@0.5:0.95':<20} {baseline_metrics.box.map:.4f}        {improved_metrics.box.map:.4f}        {improved_metrics.box.map - baseline_metrics.box.map:+.4f}      {metrics512.box.map:.4f}")
    print(f"{'精确率':<20} {baseline_metrics.box.mp:.4f}        {improved_metrics.box.mp:.4f}        {improved_metrics.box.mp - baseline_metrics.box.mp:+.4f}      {metrics512.box.mp:.4f}")
    print(f"{'召回率':<20} {baseline_metrics.box.mr:.4f}        {improved_metrics.box.mr:.4f}        {improved_metrics.box.mr - baseline_metrics.box.mr:+.4f}      {metrics512.box.mr:.4f}")
    
    return baseline_metrics, improved_metrics

In [15]:
train_improved_model()

Transferred 54/381 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.33 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\seaships\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200      3.47G      3.681      3.955      3.561          2        512: 100% ━━━━━━━━━━━━ 3559/3559 6.9it/s 8:39<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 254/254 13.9it/s 18.3s<0.1s
                   all        508       1361     0.0152      0.047    0.00506    0.00136

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200      3.71G      3.278      3.706      3.995          9        512: 0% ──────────── 1/3559 2.4it/s 0.3s<25:05

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      3.72G      2.874      2.982      2.863          3        512: 100% ━━━━━━━━━━━━ 3559/3559 7.5it/s 7:56<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 254/254 15.0it/s 17.0s<0.1s
                   all        508       1361      0.106      0.123     0.0437     0.0151

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      3.71G      2.385      4.506      2.555          1        512: 0% ──────────── 1/3559 2.3it/s 0.3s<26:14

...

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    145/200      3.72G      1.191      1.049      1.219          4        512: 100% ━━━━━━━━━━━━ 3559/3559 7.9it/s 7:33<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 254/254 15.2it/s 16.7s<0.1s
                   all        508       1361      0.667      0.471      0.528      0.299

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    146/200      3.71G       1.09     0.8425      1.324          6        512: 0% ──────────── 1/3559 2.5it/s 0.3s<24:11

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    146/200      3.72G      1.208       1.07      1.232          4        512: 100% ━━━━━━━━━━━━ 3559/3559 7.9it/s 7:32<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 254/254 15.6it/s 16.2s<0.1s
                   all        508       1361      0.664      0.474      0.535      0.301
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 116, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

146 epochs completed in 20.430 hours.
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_con\weights\last.pt, 23.7MB
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_con\weights\best.pt, 23.7MB

Validating D:\ShipTarget\ship_detection\results\improved\mmsa_con\weights\best.pt...
Ultralytics 8.4.16  Python-3.11.14 torch

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002549AECBAD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [16]:
evaluate_models()

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 738.8188.3 MB/s, size: 164.5 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\seaships\labels\test.cache... 1018 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1018/1018  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 7.1it/s 9.0s0.1s
                   all       1018       2865      0.659      0.555      0.634      0.412
Speed: 0.6ms preprocess, 4.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val6
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
custom_YOLOv11 summary (fused): 101 layers, 11,711,001 parameters, 0 gradients, 31.4 GFLOPs
val: Fast image access  (p

(ultralytics.utils.metrics.DetMetrics object with attributes:
 
 ap_class_index: array([0])
 box: ultralytics.utils.metrics.Metric object
 confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000026429439E10>
 curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
 curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
           0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        